# LSTM-arithmetic

## Dataset
- [Arithmetic dataset](https://drive.google.com/file/d/1cMuL3hF9jefka9RyF4gEBIGGeFGZYHE-/view?usp=sharing)

In [1]:
# ! pip install seaborn
# ! pip install opencc
# ! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
import opencc
import os
from sklearn.model_selection import train_test_split

data_path = './data'
torch.manual_seed(42)
np.random.seed(42)

In [2]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

,src,tgt
0,14*(43+20)=,882
1,(6+1)*5=,35
2,13+32+29=,74
3,31*(3-11)=,-248
4,24*49+1=,1177


In [3]:
# transform the input data to string
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['len'] = df_train['src'].apply(lambda x: len(x))
df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))
df_eval['len'] = df_eval['src'].apply(lambda x: len(x))
print(df_train.head())

           src   tgt  len
0  14*(43+20)=   882   11
1     (6+1)*5=    35    8
2    13+32+29=    74    9
3   31*(3-11)=  -248   10
4     24*49+1=  1177    8


# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [4]:
char_to_id = {'<pad>': 0, '<eos>': 1}
id_to_char = {0: '<pad>', 1: '<eos>'}


# write your code here
# Build a dictionary and give every token in the train dataset an id
# The dictionary should contain <eos> and <pad>
# char_to_id is to conver charactors to ids, while id_to_char is the opposite
unique_chars = set()
for text in df_train['src']:
    unique_chars.update(text)
for text in df_train['tgt']:
    unique_chars.update(text)
for text in df_eval['tgt']:
    unique_chars.update(text)

unique_chars = sorted(list(unique_chars))

for idx, char in enumerate(unique_chars):
    char_to_id[char] = idx + 2
    id_to_char[idx + 2] = char
vocab_size = len(char_to_id)

print('Vocab size{}'.format(vocab_size))
print(char_to_id)

Vocab size18
{'<pad>': 0, '<eos>': 1, '(': 2, ')': 3, '*': 4, '+': 5, '-': 6, '0': 7, '1': 8, '2': 9, '3': 10, '4': 11, '5': 12, '6': 13, '7': 14, '8': 15, '9': 16, '=': 17}


# Data Preprocessing
 - The data is processed into the format required for the model's input and output. (End with \<eos\> token)


In [ ]:
# TODO2: CORRECTED Data Preprocessing
def preprocess_row(row):
    src = row['src']
    tgt = row['tgt']
    
    # Find the position of '='
    equal_pos = src.index('=')
    
    # Create the full sequence: src + tgt
    full_sequence = src + tgt 
    
    # Convert to IDs
    char_id_list = [char_to_id[char] for char in full_sequence]
    
    label_id_list = char_id_list[1:] + [char_to_id['<eos>']]
    
    for i in range(equal_pos):  # Changed from equal_pos + 1
        label_id_list[i] = char_to_id['<pad>']
    
    return pd.Series({
        'char_id_list': char_id_list,
        'label_id_list': label_id_list
    })

df_train[['char_id_list', 'label_id_list']] = df_train.apply(preprocess_row, axis=1)
df_eval[['char_id_list', 'label_id_list']] = df_eval.apply(preprocess_row, axis=1)

In [6]:
print(char_to_id)

{'<pad>': 0, '<eos>': 1, '(': 2, ')': 3, '*': 4, '+': 5, '-': 6, '0': 7, '1': 8, '2': 9, '3': 10, '4': 11, '5': 12, '6': 13, '7': 14, '8': 15, '9': 16, '=': 17}


# Hyper Parameters

|Hyperparameter|Meaning|Value|
|-|-|-|
|`batch_size`|Number of data samples in a single batch|64|
|`epochs`|Total number of epochs to train|10|
|`embed_dim`|Dimension of the word embeddings|256|
|`hidden_dim`|Dimension of the hidden state in each timestep of the LSTM|256|
|`lr`|Learning Rate|0.001|
|`grad_clip`|To prevent gradient explosion in RNNs, restrict the gradient range|1|

In [7]:
batch_size = 128
epochs = 10
embed_dim = 256
hidden_dim = 256
lr = 0.001
grad_clip = 1

# Data Batching
- Use `torch.utils.data.Dataset` to create a data generation tool called  `dataset`.
- The, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.

- Example: 1+2-3=0
    - Model input: 1 + 2 - 3 = 0
    - Model output: / / / / / 0 &lt;eos&gt;  (the '/' can be replaced with &lt;pad&gt;)
    - The key for the model's output is that the model does not need to predict the next character of the previous part. What matters is that once the model sees '=', it should start generating the answer, which is '0'. After generating the answer, it should also generate&lt;eos&gt;

In [8]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    
    def __len__(self):
        # return the amount of data
        return len(self.sequences)
    
    def __getitem__(self, index):
        # Extract the input data x and the ground truth y from the data
        x = self.sequences.iloc[index]['char_id_list']
        y = self.sequences.iloc[index]['label_id_list']
        return x, y

# collate function, used to build dataloader
def collate_fn(batch):
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])
    
    # Pad the input sequence
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])
    
    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])
    
    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens

In [9]:
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])
ds_eval = Dataset(df_eval[['char_id_list', 'label_id_list']])

In [10]:
# Build dataloader of train set and eval set, collate_fn is the collate function
dl_train = torch.utils.data.DataLoader(ds_train,
                                       batch_size=batch_size,
                                       shuffle=True,
                                       collate_fn=collate_fn)
dl_eval = torch.utils.data.DataLoader(ds_eval,
                                      batch_size=batch_size,
                                      shuffle=False,
                                      collate_fn=collate_fn)

# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

In [ ]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()
        
        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                                            embedding_dim=embed_dim,
                                            padding_idx=char_to_id['<pad>'])
        
        self.rnn_layer1 = torch.nn.LSTM(input_size=embed_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        
        self.rnn_layer2 = torch.nn.LSTM(input_size=hidden_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        
        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=hidden_dim),
                                          torch.nn.ReLU(),
                                          torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=vocab_size))
        
    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)
    
    # The forward pass of the model
    def encoder(self, batch_x, batch_x_lens):
        batch_x = self.embedding(batch_x)
        
        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                                          batch_x_lens,
                                                          batch_first=True,
                                                          enforce_sorted=False)
        
        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)
        
        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,
                                                            batch_first=True)
        
        batch_x = self.linear(batch_x)
        
        return batch_x
    
    def generator(self, start_char, max_len=50):
        """Generate answer given an arithmetic expression ending with '='"""
        self.eval()
        device = next(self.parameters()).device

        # Convert input string to tensor
        char_ids = [char_to_id[c] for c in start_char]
        input_tensor = torch.tensor([char_ids], dtype=torch.long).to(device)

        generated_chars = []

        with torch.no_grad():
            embedded = self.embedding(input_tensor)
            output1, (h1, c1) = self.rnn_layer1(embedded)
            output2, (h2, c2) = self.rnn_layer2(output1)

            last_hidden = output2[:, -1:, :]  # Shape: [1, 1, hidden_dim]
            logits = self.linear(last_hidden)
            next_char_id = torch.argmax(logits, dim=-1).item()

            while next_char_id != char_to_id['<eos>'] and len(generated_chars) < max_len:
                generated_chars.append(id_to_char[next_char_id])

                # Prepare next input
                next_input = torch.tensor([[next_char_id]], dtype=torch.long).to(device)
                embedded_next = self.embedding(next_input)

                # Pass through LSTM layers with previous hidden states
                output1, (h1, c1) = self.rnn_layer1(embedded_next, (h1, c1))
                output2, (h2, c2) = self.rnn_layer2(output1, (h2, c2))

                # Predict next character
                logits = self.linear(output2)
                next_char_id = torch.argmax(logits, dim=-1).item()

        return generated_chars  # Return only the generated answer

In [12]:
torch.manual_seed(2)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim)

cuda


In [13]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>'])
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Training
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss `loss`
        3. Use `loss.backward` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_` to limit the gradient values between `-grad_clip` &lt; and &lt; `grad_clip`.
        5. Use `optimizer.step()` to update the model (backpropagation).
2.  After every `1000` batches, output the current loss to monitor whether it is converging.

In [14]:
from tqdm import tqdm

model = model.to(device)
best_accuracy = 0

for epoch in range(1, epochs + 1):
    # ===== TRAINING =====
    model.train()
    train_loss = 0
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # Forward pass
        batch_pred_y = model(batch_x, batch_x_lens)
        
        loss = criterion(
            batch_pred_y.reshape(-1, vocab_size),
            batch_y.reshape(-1)
        )
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        
        train_loss += loss.item()
        bar.set_postfix(loss=loss.item())
    
    avg_train_loss = train_loss / len(dl_train)
    
    model.eval()
    matched = 0
    total = 0
    
    # Sample from eval set
    eval_sample_size = min(10000, len(df_eval))
    df_eval_sample = df_eval.sample(n=eval_sample_size, random_state=epoch)
    
    bar_eval = tqdm(
        df_eval_sample.itertuples(index=False),
        total=len(df_eval_sample),
        desc=f"Validation epoch {epoch}"
    )
    
    with torch.no_grad():
        for idx, row in enumerate(bar_eval):
            input_expr = row.src  # e.g., "1+2="
            expected_answer = row.tgt  # e.g., "3"
            
            # Generate answer
            generated_chars = model.generator(input_expr)
            predicted_answer = ''.join(generated_chars)
            
            # Check exact match
            if predicted_answer == expected_answer:
                matched += 1
            
            total += 1
            
            # Show first 5 examples
            if idx < 5:
                print(f"\nInput: {input_expr}")
                print(f"Generated: {predicted_answer}")
                print(f"Expected: {expected_answer}")
                print(f"Match: {predicted_answer == expected_answer}")
            
            # Update progress bar
            current_acc = matched / total
            bar_eval.set_postfix(EM=f"{current_acc:.4f}")
    
    # Calculate final accuracy
    accuracy = matched / total
    print(f"\nEpoch {epoch} Summary:")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Eval Accuracy (EM): {accuracy:.4f}")
    
    # Save best model
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  ✓ New best model saved! (Accuracy: {best_accuracy:.4f})")

print(f"\nTraining complete! Best accuracy: {best_accuracy:.4f}")

Validation epoch 1:   1%|          | 62/10000 [00:00<00:16, 611.88it/s, EM=0.5596]


Input: (10+41)-14=
Generated: 37
Expected: 37
Match: True

Input: (30-18)*20=
Generated: 240
Expected: 240
Match: True

Input: 49-16+43=
Generated: 76
Expected: 76
Match: True

Input: 37*27+40=
Generated: 1031
Expected: 1039
Match: False

Input: (11+27)*5=
Generated: 190
Expected: 190
Match: True


Validation epoch 1: 100%|██████████| 10000/10000 [00:16<00:00, 600.22it/s, EM=0.6080]



Epoch 1 Summary:
  Train Loss: 0.4980
  Eval Accuracy (EM): 0.6080
  ✓ New best model saved! (Accuracy: 0.6080)


Validation epoch 2:   1%|          | 118/10000 [00:00<00:16, 584.43it/s, EM=0.6780]

...


Epoch 9 Summary:
  Train Loss: 0.0795
  Eval Accuracy (EM): 0.8874
  ✓ New best model saved! (Accuracy: 0.8874)


Validation epoch 10:   1%|          | 61/10000 [00:00<00:16, 604.28it/s, EM=0.9113]


Input: 29+(39*15)=
Generated: 614
Expected: 614
Match: True

Input: (26*22)+20=
Generated: 592
Expected: 592
Match: True

Input: (19*49)-33=
Generated: 898
Expected: 898
Match: True

Input: 3*16*30=
Generated: 1440
Expected: 1440
Match: True

Input: 32+1-14=
Generated: 19
Expected: 19
Match: True


Validation epoch 10: 100%|██████████| 10000/10000 [00:17<00:00, 575.46it/s, EM=0.8819]


Epoch 10 Summary:
  Train Loss: 0.0747
  Eval Accuracy (EM): 0.8819

Training complete! Best accuracy: 0.8874
